# Feature extraction: single-object trials and spikes
Interact with `scripts/extract.py`. Trials are the 150 clean single-object trials; units are all 137 sorted units with their per-trial observation mask.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists():
    root = root.parent
sys.path.insert(0, str(root / 'scripts'))

import numpy as np
import pandas as pd
from extract import load_single_object_trials, load_units

In [ ]:
trials = load_single_object_trials()
print(trials.shape)
trials.head()

In [ ]:
# identity x position design (3 x 3, unbalanced)
pd.crosstab(trials.identity, trials.position)

In [ ]:
units, obs = load_units()
print(len(units), 'units')
units.metadata.head()

In [ ]:
# obs mask: fraction of the single-object trials each unit was recorded in.
# A 0 spike-count in an unobserved trial is missing data, not silence.
coverage = obs[trials.index].mean(axis=1)
print('fully observed:', int((coverage == 1).sum()), '/', len(coverage))
coverage.describe()

In [ ]:
# Sanity: bin one unit around stim onset for one condition (0-2 s window).
import matplotlib.pyplot as plt

sel = trials[(trials.identity == 'a') & (trials.position == 0)]
obs_units = obs[sel.index].all(axis=1)
uid = coverage[obs_units].index[0]

edges = np.arange(0, 2.001, 0.1)
psth = np.zeros(len(edges) - 1)
spk = units[uid].t
for t0 in sel.stim_time:
    psth += np.histogram(spk - t0, bins=edges)[0]
psth = psth / len(sel) / 0.1

plt.plot(edges[:-1], psth)
plt.axvline(1.0, ls='--', c='k')  # delay onset
plt.xlabel('time from stim onset (s)'); plt.ylabel('rate (Hz)')
plt.title(f'unit {uid}, identity a @ pos 0');

## Mean activity per neuron, marginalised

For each neuron: PSTH (0-2 s from stim onset, 100 ms bins) averaged within each identity
(marginalising over position) and within each position (marginalising over identity).
Each neuron uses only the trials in which it was observed.

In [ ]:
centers = edges[:-1] + 0.05  # bin centres of the 0-2 s window


def marginal_psths(uid):
    """Return (identity means {a,b,c}, position means {0,1,2}) over this unit's observed trials."""
    sub = trials.loc[trials.index[obs.loc[uid, trials.index].to_numpy()]]
    spk = units[uid].t
    counts = np.stack([np.histogram(spk - t0, bins=edges)[0] for t0 in sub.stim_time]) / 0.1
    id_means = {k: counts[(sub.identity == k).to_numpy()].mean(0) for k in ['a', 'b', 'c']}
    pos_means = {k: counts[(sub.position == k).to_numpy()].mean(0) for k in [0, 1, 2]}
    return id_means, pos_means

In [ ]:
psths = {uid: marginal_psths(uid) for uid in units.keys()}
ncol = 12
nrow = int(np.ceil(len(units) / ncol))

for kind, keys, colors in [('identity', ['a', 'b', 'c'], ['C0', 'C1', 'C2']),
                           ('position', [0, 1, 2], ['C3', 'C4', 'C5'])]:
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.4, nrow * 1.1), sharex=True)
    axes = axes.ravel()
    for i, uid in enumerate(units.keys()):
        means = psths[uid][0 if kind == 'identity' else 1]
        for k, c in zip(keys, colors):
            axes[i].plot(centers, means[k], c=c, lw=0.7)
        axes[i].axvline(1.0, ls='--', c='k', lw=0.4)  # delay onset
        axes[i].set_title(str(uid), fontsize=5, pad=1)
        axes[i].tick_params(labelsize=4, length=1)
    for j in range(len(units), len(axes)):
        axes[j].axis('off')
    over = 'position' if kind == 'identity' else 'identity'
    fig.legend(handles=[plt.Line2D([], [], c=c, label=str(k)) for k, c in zip(keys, colors)],
               loc='upper right', fontsize=7, title=f'marginal over {over}')
    fig.supxlabel('time from stim onset (s)', fontsize=8)
    fig.supylabel('rate (Hz)', fontsize=8)
    fig.tight_layout()

## Population heatmaps per condition

Six neuron x time heatmaps: one per identity (marginal over position) and one per position
(marginal over identity). Each neuron is z-scored across the pooled bins of its three conditions
so the colour scale is shared within a row.

`plot_condition_heatmaps(sort_by=...)` controls the neuron ordering:
- `None` (default): each row is sorted by peak time of its own condition-averaged response.
- `('identity', 'a')` or `('position', 2)`: all six maps use one shared order, from peak time
  in that reference condition. This shows whether the reference's sequence persists across
  the other conditions.

In [ ]:
uids = list(units.keys())


def zscored_maps(which):
    """Per-condition z-scored neuron x time maps for one marginalisation (0=identity, 1=position)."""
    keys = ['a', 'b', 'c'] if which == 0 else [0, 1, 2]
    mats = {k: np.stack([psths[u][which][k] for u in uids]) for k in keys}
    pooled = np.concatenate([mats[k] for k in keys], axis=1)
    mu = np.nanmean(pooled, axis=1, keepdims=True)
    sd = np.nanstd(pooled, axis=1, keepdims=True)
    sd[sd == 0] = np.nan
    return keys, {k: (mats[k] - mu) / sd for k in keys}


def _peak_order(mat):
    """Neuron order by peak-time of each row; all-nan neurons go last."""
    peak = np.full(mat.shape[0], np.nan)
    valid = ~np.all(np.isnan(mat), axis=1)
    peak[valid] = np.nanargmax(mat[valid], axis=1)
    return np.argsort(np.where(np.isnan(peak), np.inf, peak))


def plot_condition_heatmaps(sort_by=None):
    """sort_by=None sorts each row by its own average; ('identity','a')/('position',2) sorts all six by that condition."""
    kinds = {'identity': 0, 'position': 1}
    maps = {w: zscored_maps(w) for w in (0, 1)}
    shared_order = None
    if sort_by is not None:
        _, sz = maps[kinds[sort_by[0]]]
        shared_order = _peak_order(sz[sort_by[1]])

    fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharex=True, sharey=True)
    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad('lightgray')
    for r, which in enumerate([0, 1]):
        keys, z = maps[which]
        label = 'identity' if which == 0 else 'position'
        order = (shared_order if shared_order is not None
                 else _peak_order(np.nanmean(np.stack([z[k] for k in keys]), axis=0)))
        for cidx, k in enumerate(keys):
            ax = axes[r, cidx]
            im = ax.imshow(z[k][order], aspect='auto', cmap=cmap, vmin=-2, vmax=2,
                           extent=[0, 2, len(uids), 0], interpolation='nearest')
            ax.axvline(1.0, ls='--', c='k', lw=0.6)  # delay onset
            ax.set_title(f'{label} {k}', fontsize=9)
            if r == 1:
                ax.set_xlabel('time from stim onset (s)')
            if cidx == 0:
                ax.set_ylabel('neuron')
    fig.suptitle(f'sorted by: {"per-row average" if sort_by is None else f"{sort_by[0]} {sort_by[1]}"}', fontsize=10)
    fig.colorbar(im, ax=axes, shrink=0.6, label='z-scored rate')


plot_condition_heatmaps()

In [ ]:
# All six maps sorted by peak time in one reference condition.
plot_condition_heatmaps(sort_by=('position', 2))

## Per-session counts across the dandiset

Unit and trial counts for all 47 sessions (DANDI:000620/0.260127.2208), streamed from the archive.
`held_95` = units with obs_trials coverage >= 0.95; `single_object_success` = single-object trials
with reward_duration > 0.

In [ ]:
session_counts = pd.read_csv(root / 'session_counts.csv')
print(session_counts.shape)
session_counts[session_counts.three_position == True].sort_values('dmfc_good', ascending=False).head(10)

### Candidate sessions

**DMFC**

Perle 
- 2022-06-12

**FEF**

Perle
- 2022-06-04 (45 good units, 120 trials)
- 2022-05-29 (41 good units, 147 trials)
- 2022-06-16 (33 good units, 200 trials)